# Predicting Players' Performance in T20 Cricket

**Part A** replicates *"Predicting players' performance in T20 cricket matches using Machine Learning approaches"* (Thilakarathne, 2024) on `ball_by_ball_it20.csv`. The paper trains Linear Regression and Random Forest. We add XGBoost as a new model.

**Part B** is the new contribution: a **live chase win probability model** (like the win bar on TV) with **explainable AI**, so every prediction comes with the reasons behind it. Both parts are exported and served by the Flask and Angular app.

Metrics follow the paper (Table 4): RSE (R squared) and MSE on train and test splits.

## 0. Setup

In [ ]:
%pip install -q shap

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import xgboost as xgb
from pathlib import Path

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (mean_squared_error, r2_score, accuracy_score,
                             roc_auc_score, log_loss, brier_score_loss)
from xgboost import XGBRegressor, XGBClassifier

pd.set_option('display.max_columns', None)
np.random.seed(42)

# Part A: Replication of the paper, plus XGBoost

## 1. Load and clean the data

In [ ]:
df = pd.read_csv('/content/ball_by_ball_it20.csv', index_col=0)
print(df.shape)
df.head()

In [ ]:
# drop duplicates and rows with no ball information
df = df.drop_duplicates()
df = df.dropna(subset=['Match ID', 'Batter', 'Bowler', 'Runs From Ball']).copy()

# a missing Valid Ball flag is treated as a normal legal delivery
df['Valid Ball'] = df['Valid Ball'].fillna(1)
print(df.shape)

## 2. Batting table (one row per batter per innings)

The raw file is ball by ball, while the paper used a table already summed per player per match. We rebuild that table here.

In [ ]:
bat_group = df.groupby(['Match ID', 'Innings', 'Batter'], as_index=False)

batting = bat_group.agg(
    Team=('Bat First', 'first'),
    Opposition=('Bat Second', 'first'),
    Runs=('Batter Runs', 'sum'),
    Balls=('Valid Ball', 'sum'),
    Fours=('Runs From Ball', lambda x: (x == 4).sum()),
    Sixes=('Runs From Ball', lambda x: (x == 6).sum()),
)

# how each batter got out (if at all)
dismissals = df[df['Wicket'] == 1][['Match ID', 'Innings', 'Player Out', 'Method', 'Bowler']]
dismissals = dismissals.rename(columns={'Player Out': 'Batter',
                                        'Method': 'Wicket Type',
                                        'Bowler': 'Dismissal Bowler'})
dismissals = dismissals.drop_duplicates(subset=['Match ID', 'Innings', 'Batter'])

batting = batting.merge(dismissals, on=['Match ID', 'Innings', 'Batter'], how='left')
batting['Is Out'] = batting['Wicket Type'].notna().astype(int)
batting['Wicket Type'] = batting['Wicket Type'].fillna('Not Out')

batting = batting[batting['Balls'] > 0].copy()
batting['Strike Rate'] = batting['Runs'] / batting['Balls'] * 100

print(batting.shape)
batting.head()

## 3. Bowling table (one row per bowler per innings)

In [ ]:
bowl_group = df.groupby(['Match ID', 'Innings', 'Bowler'], as_index=False)

bowling = bowl_group.agg(
    Team=('Bat Second', 'first'),
    Opposition=('Bat First', 'first'),
    Balls=('Valid Ball', 'sum'),
    Conceded=('Bowler Runs Conceded', 'sum'),
    Dots=('Runs From Ball', lambda x: (x == 0).sum()),
    Fours=('Runs From Ball', lambda x: (x == 4).sum()),
    Sixers=('Runs From Ball', lambda x: (x == 6).sum()),
)

# wickets credited to the bowler (run outs do not count)
wkts = df[(df['Wicket'] == 1) & (df['Method'] != 'run out')]
wkts = wkts.groupby(['Match ID', 'Innings', 'Bowler']).size().reset_index(name='Wickets')
bowling = bowling.merge(wkts, on=['Match ID', 'Innings', 'Bowler'], how='left')
bowling['Wickets'] = bowling['Wickets'].fillna(0)

bowling = bowling[bowling['Balls'] > 0].copy()
bowling['Overs'] = bowling['Balls'] / 6
bowling['Economy'] = bowling['Conceded'] / bowling['Overs']

print(bowling.shape)
bowling.head()

## 4. Pre-processing (mean filling and label encoding, as in the paper)

In [ ]:
def preprocess(frame, numeric_cols, cat_cols):
    frame = frame.copy()
    for c in numeric_cols:
        frame[c] = frame[c].fillna(frame[c].mean())
    for c in cat_cols:
        frame[c + '_encoded'] = LabelEncoder().fit_transform(frame[c].astype(str))
    frame = frame.replace([np.inf, -np.inf], np.nan)
    return frame.dropna(subset=numeric_cols)


batting_p = preprocess(batting, ['Runs', 'Balls', 'Fours', 'Sixes', 'Strike Rate'],
                       ['Team', 'Opposition', 'Wicket Type'])
bowling_p = preprocess(bowling, ['Conceded', 'Balls', 'Fours', 'Sixers', 'Economy', 'Wickets'],
                       ['Team', 'Opposition'])
print(batting_p.shape, bowling_p.shape)

## 5. Features and train/test split

The paper picks its predictors with PCA and correlation analysis. We use the same predictor sets directly.

* Batting: predict **Runs** from Balls, Fours, Sixes, Strike Rate
* Bowling: predict **Conceded** from Fours, Sixers, Balls, Economy

In [ ]:
X_bat = batting_p[['Balls', 'Fours', 'Sixes', 'Strike Rate']]
y_bat = batting_p['Runs']

X_bowl = bowling_p[['Fours', 'Sixers', 'Balls', 'Economy']]
y_bowl = bowling_p['Conceded']

Xb_train, Xb_test, yb_train, yb_test = train_test_split(X_bat, y_bat, test_size=0.2, random_state=42)
Xw_train, Xw_test, yw_train, yw_test = train_test_split(X_bowl, y_bowl, test_size=0.2, random_state=42)

## 6. Train Linear Regression, Random Forest and XGBoost

Linear Regression and Random Forest are the paper's models. XGBoost is the new one. We keep every fitted model so the XGBoost ones can be exported later without retraining.

In [ ]:
def evaluate(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)
    return {
        'RSE_train': r2_score(y_train, pred_train),
        'MSE_train': mean_squared_error(y_train, pred_train),
        'RSE_test': r2_score(y_test, pred_test),
        'MSE_test': mean_squared_error(y_test, pred_test),
    }


def make_models():
    return {
        'Linear Regression': LinearRegression(),
        'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42),
        'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42),
    }


splits = {
    'Batting': (Xb_train, yb_train, Xb_test, yb_test),
    'Bowling': (Xw_train, yw_train, Xw_test, yw_test),
}

results = {}
fitted = {}
for section, (X_tr, y_tr, X_te, y_te) in splits.items():
    for name, model in make_models().items():
        results[(section, name)] = evaluate(model, X_tr, y_tr, X_te, y_te)
        fitted[(section, name)] = model

results_df = pd.DataFrame(results).T[['RSE_train', 'MSE_train', 'RSE_test', 'MSE_test']].round(4)
results_df.index.names = ['Section', 'Algorithm']
display(results_df)

print('Best model per section (highest test RSE):')
for section in ['Batting', 'Bowling']:
    sec = results_df.loc[section]
    best = sec['RSE_test'].idxmax()
    print(f"  {section}: {best} (test RSE {sec.loc[best, 'RSE_test']}, test MSE {sec.loc[best, 'MSE_test']})")

## 7. Comparison with the paper (Table 4)

The paper's numbers are typed in below. A positive RSE difference means our model explains more variance. A positive MSE difference means our error is larger. Differences are expected because the paper used a pre-aggregated Kaggle table, while we rebuild the table from raw ball by ball data.

In [ ]:
paper_results = {
    ('Batting', 'Linear Regression'): {'RSE_train': 0.9752, 'MSE_train': 6.2218, 'RSE_test': 0.9551, 'MSE_test': 6.6149},
    ('Batting', 'Random Forest'):     {'RSE_train': 0.9873, 'MSE_train': 0.9965, 'RSE_test': 0.9941, 'MSE_test': 1.5066},
    ('Bowling', 'Linear Regression'): {'RSE_train': 0.9015, 'MSE_train': 10.9456, 'RSE_test': 0.8898, 'MSE_test': 9.8721},
    ('Bowling', 'Random Forest'):     {'RSE_train': 0.9456, 'MSE_train': 0.0054, 'RSE_test': 0.9098, 'MSE_test': 5.1478},
}
paper_df = pd.DataFrame(paper_results).T[['RSE_train', 'MSE_train', 'RSE_test', 'MSE_test']]
paper_df.index.names = ['Section', 'Algorithm']

ours_df = results_df.loc[paper_df.index]
comparison = pd.concat(
    {'Paper': paper_df, 'Ours': ours_df, 'Diff (Ours minus Paper)': ours_df - paper_df},
    axis=1,
).round(4)
comparison

## 8. A weakness we noticed, and why Part B exists

Strike Rate is `Runs / Balls * 100` and Economy is `Conceded / Overs`. Each of these predictors already contains the target, so R squared near 0.999 is partly built in. The models are accurate, but they do not forecast anything we could not already compute.

Part B fixes this by predicting an outcome (who wins the chase) from the match situation alone.

# Part B: Live chase win probability with explainable AI

**Question:** the team batting second needs a target. At any ball of the chase, what is their chance of winning?

**Inputs (all known at that moment):** target, current score, wickets lost, balls bowled.
From these we derive runs needed, balls remaining, wickets in hand, current run rate and required run rate.

**Why this is safer than Part A:** no input contains the answer. We also split train and test **by match**, not by row, because the 100+ rows of one chase are strongly related and a row based split would leak.

## 9. Build the chase dataset (second innings only)

In [ ]:
chase = df[df['Innings'] == 2].copy()

# keep full 20 over chases only, the app assumes 120 balls
first_ball_left = chase.groupby('Match ID')['Balls Remaining'].transform('max')
chase = chase[first_ball_left >= 119].copy()

chase = chase.dropna(subset=['Target Score', 'Innings Runs', 'Innings Wickets',
                             'Balls Remaining', 'Chased Successfully'])
chase['Balls Bowled'] = 120 - chase['Balls Remaining']

# keep live situations only (chase not finished yet)
live = ((chase['Target Score'] - chase['Innings Runs'] > 0)
        & (chase['Balls Remaining'] > 0)
        & (chase['Innings Wickets'] < 10))
chase = chase[live].copy()

print('rows:', len(chase), '| matches:', chase['Match ID'].nunique())
print('chasing team won:', round(chase.groupby('Match ID')['Chased Successfully'].first().mean() * 100, 1), '% of matches')

In [ ]:
# Target Score is left out on purpose, see the note in section 10.
FEATURES = ['Runs Needed', 'Balls Remaining', 'Wickets In Hand',
            'Current Run Rate', 'Required Run Rate']


def make_features(target, score, wickets, balls_bowled):
    # Used for training here and copied into the Flask app, so both sides agree.
    target, score, wickets, balls_bowled = (np.atleast_1d(np.asarray(v, dtype=float))
                                            for v in (target, score, wickets, balls_bowled))
    runs_needed = target - score
    balls_left = 120 - balls_bowled
    return pd.DataFrame({
        'Runs Needed': runs_needed,
        'Balls Remaining': balls_left,
        'Wickets In Hand': 10 - wickets,
        'Current Run Rate': np.where(balls_bowled > 0, score * 6 / np.maximum(balls_bowled, 1), 0.0),
        'Required Run Rate': runs_needed * 6 / np.maximum(balls_left, 1),
    })[FEATURES]


X_win = make_features(chase['Target Score'].to_numpy(), chase['Innings Runs'].to_numpy(),
                      chase['Innings Wickets'].to_numpy(), chase['Balls Bowled'].to_numpy())
y_win = chase['Chased Successfully'].astype(int).to_numpy()
match_ids = chase['Match ID'].to_numpy()
X_win.head()

## 10. Split by match, then train

**Why Target Score is not a model input.** It is the same number for every ball of a chase, so a tree model can use it like a match ID and memorise which team won. Our first version did exactly that: it scored about 57% accuracy on unseen matches while a plain logistic regression scored about 88%. Runs needed already carries the target information in a way that changes ball by ball, so nothing useful is lost.

**Other guards against memorising:** shallow trees, large minimum leaf size, and monotone rules (more runs needed can never help the chasing team, more wickets in hand can never hurt it).

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X_win, y_win, groups=match_ids))

Xw_tr, Xw_te = X_win.iloc[train_idx], X_win.iloc[test_idx]
yw_tr, yw_te = y_win[train_idx], y_win[test_idx]
print('train matches:', len(set(match_ids[train_idx])), '| test matches:', len(set(match_ids[test_idx])))

# Order follows FEATURES for the monotone rules: runs needed (down), balls remaining (free),
# wickets in hand (up), current run rate (free), required run rate (down).
win_model = XGBClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=3,
    min_child_weight=50, subsample=0.8, colsample_bytree=0.8, reg_lambda=5,
    monotone_constraints=(-1, 0, 1, 0, -1),
    eval_metric='logloss', random_state=42,
)
win_model.fit(Xw_tr, yw_tr)

# a simple logistic regression as the baseline to beat
baseline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
baseline.fit(Xw_tr, yw_tr)


def classification_scores(model):
    p = model.predict_proba(Xw_te)[:, 1]
    return {
        'Accuracy': accuracy_score(yw_te, p > 0.5),
        'ROC AUC': roc_auc_score(yw_te, p),
        'Log loss': log_loss(yw_te, p),
        'Brier score': brier_score_loss(yw_te, p),
        'Train log loss': log_loss(yw_tr, model.predict_proba(Xw_tr)[:, 1]),
    }


win_results = pd.DataFrame({
    'Logistic Regression (baseline)': classification_scores(baseline),
    'XGBoost (win model)': classification_scores(win_model),
}).T.round(4)
win_results

How to read it: accuracy and ROC AUC are better when higher. Log loss and Brier score measure how honest the percentages are, and they are better when lower. A model that says 90% should win about 90% of the time. If train log loss is far below test log loss, the model is memorising.

## 10b. Data check

These checks confirm the labels and inputs behave as expected: the label matches how each chase really ended, `Runs to Get` equals target minus runs, each input carries signal on its own, and the scores hold up on five different match splits. Run it whenever the numbers above look strange.

In [ ]:
# DIAGNOSTIC: is the chase data behaving as expected? (paste after the training cell)
from sklearn.model_selection import GroupKFold, cross_val_score

print('0. rows train/test:', len(train_idx), len(test_idx),
      '| matches shared by both:', len(set(match_ids[train_idx]) & set(match_ids[test_idx])))

c2 = df[df['Innings'] == 2]
last = c2.groupby('Match ID').tail(1)
reached = (last['Innings Runs'] >= last['Target Score']).astype(int)
won = last['Chased Successfully'].astype(int)
print('1. label agrees with how the chase ended:', round((reached == won).mean(), 3))
print('2. label agrees with Winner == Bat Second:',
      round(((last['Winner'] == last['Bat Second']).astype(int) == won).mean(), 3))

gap = c2['Runs to Get'] - (c2['Target Score'] - c2['Innings Runs'])
print('3. Runs to Get minus (Target minus Runs), summary:')
print(gap.describe().round(2).to_dict())

print('4. Balls Remaining on first row of a chase:',
      c2.groupby('Match ID')['Balls Remaining'].first().describe().round(1).to_dict())

print('5. chasing team won share, train vs test:',
      round(y_win[train_idx].mean(), 3), round(y_win[test_idx].mean(), 3))

print('6. AUC of each input on its own (0.5 means no signal):')
for col in FEATURES:
    print(f'   {col:<18}', round(roc_auc_score(y_win, X_win[col]), 3))
print(f"   {'Target Score':<18}", round(roc_auc_score(y_win, chase['Target Score']), 3))

# which test matches does the baseline get most wrong?
p = baseline.predict_proba(Xw_te)[:, 1]
t = pd.DataFrame({'match': match_ids[test_idx], 'label': yw_te, 'p': p})
t['ll'] = -(t['label'] * np.log(np.clip(t['p'], 1e-6, 1)) + (1 - t['label']) * np.log(np.clip(1 - t['p'], 1e-6, 1)))
worst = t.groupby('match').agg(rows=('ll', 'size'), label=('label', 'first'),
                               avg_p=('p', 'mean'), log_loss=('ll', 'mean'))
print('7. worst test matches for the baseline:')
display(worst.sort_values('log_loss', ascending=False).head(8).round(3))

# is the train/test gap a split problem? score the same baseline on 5 different match splits
cv = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
                     X_win, y_win, groups=match_ids, cv=GroupKFold(5), scoring='neg_log_loss')
print('8. baseline log loss on 5 match based folds:', (-cv).round(3))

## 11. The win probability graph for one unseen match

This is the demo graph. Dots mark wickets. The line falls after wickets and rises after boundaries.

In [ ]:
test_rows = chase.iloc[test_idx]
match_id = test_rows.groupby('Match ID').size().idxmax()      # longest chase in the test set
mask = (test_rows['Match ID'] == match_id).to_numpy()

prob = win_model.predict_proba(Xw_te[mask])[:, 1] * 100
overs = test_rows.loc[mask, 'Balls Bowled'].to_numpy() / 6
wicket_fell = test_rows.loc[mask, 'Innings Wickets'].diff().fillna(0).to_numpy() > 0
info = test_rows[mask].iloc[0]

plt.figure(figsize=(9, 4))
plt.plot(overs, prob, label='Win probability of chasing team')
plt.scatter(overs[wicket_fell], prob[wicket_fell], zorder=3, label='Wicket')
plt.axhline(50, linestyle='--', linewidth=1)
plt.ylim(0, 100)
plt.xlabel('Overs bowled')
plt.ylabel('Win probability (%)')
plt.title(f"{info['Bat Second']} chasing {int(info['Target Score'])} against {info['Bat First']}")
plt.legend()
plt.tight_layout()
plt.show()

## 12. Explainable AI: why did the model say that?

SHAP gives each input a share of the credit for a prediction. A positive value pushes the win chance up and a negative value pushes it down. The values add up to the gap between this prediction and the average prediction.

**Global view:** which inputs matter most across many situations.

In [ ]:
explainer = shap.TreeExplainer(win_model)
sample = Xw_te.sample(min(2000, len(Xw_te)), random_state=42)
shap_values = explainer.shap_values(sample)

shap.summary_plot(shap_values, sample, plot_type='bar', show=False)
plt.title('Average impact of each input on the win prediction')
plt.tight_layout()
plt.show()

In [ ]:
# beeswarm: each dot is one situation, colour shows if the input value was high or low
shap.summary_plot(shap_values, sample)

**Single prediction view:** this is what the app shows under the result.

XGBoost can return these SHAP values itself (`pred_contribs=True`), so the Flask backend needs no extra library. The raw values are in log odds, so we convert them into percentage points to make them easy to read.

In [ ]:
def contributions_to_points(values, bias):
    # raw SHAP values are in log odds; rescale them into win probability points
    sigmoid = lambda z: 1 / (1 + np.exp(-z))
    total = float(np.sum(values))
    p, base = sigmoid(bias + total), sigmoid(bias)
    scale = (p - base) / total if abs(total) > 1e-9 else p * (1 - p)
    return np.asarray(values) * scale * 100, base * 100, p * 100


def explain_state(target, score, wickets, balls_bowled):
    frame = make_features(target, score, wickets, balls_bowled)
    raw = win_model.get_booster().predict(xgb.DMatrix(frame), pred_contribs=True)[0]
    points, base, prob = contributions_to_points(raw[:-1], raw[-1])

    print(f'Chasing {target}: {score}/{wickets} after {balls_bowled // 6}.{balls_bowled % 6} overs')
    print(f'Win chance for the chasing team: {prob:.1f}%   (average situation: {base:.1f}%)')
    for i in np.argsort(-np.abs(points)):
        print(f'  {frame.columns[i]:<18} = {frame.iloc[0, i]:>7.2f}   {points[i]:+6.1f} points')
    print()


explain_state(target=180, score=95, wickets=2, balls_bowled=66)    # chase on track
explain_state(target=180, score=95, wickets=6, balls_bowled=90)    # collapse, far behind
explain_state(target=150, score=140, wickets=7, balls_bowled=108)  # 10 needed off 12 balls

Quick correctness check. The SHAP values plus the base value must rebuild the model output exactly, and the values from `pred_contribs` must match the `shap` library.

In [ ]:
check = Xw_te.iloc[:200]
raw = win_model.get_booster().predict(xgb.DMatrix(check), pred_contribs=True)
margin = win_model.get_booster().predict(xgb.DMatrix(check), output_margin=True)

assert np.allclose(raw.sum(axis=1), margin, atol=1e-4), 'contributions do not add up to the model output'
assert np.allclose(raw[:, :-1], explainer.shap_values(check), atol=1e-4), 'differs from the shap library'
print('OK: contributions add up to the prediction and match the shap library')

## 13. Export the models for the Flask backend

Three files are created: the two original models (XGBoost versions of batting and bowling) and the new win probability model. The backend expects these exact column names in this order.

In [ ]:
EXPORT_DIR = Path('models_export')
EXPORT_DIR.mkdir(exist_ok=True)

assert list(Xb_train.columns) == ['Balls', 'Fours', 'Sixes', 'Strike Rate']
assert list(Xw_train.columns) == ['Fours', 'Sixers', 'Balls', 'Economy']
assert list(Xw_tr.columns) == ['Runs Needed', 'Balls Remaining', 'Wickets In Hand',
                               'Current Run Rate', 'Required Run Rate']

fitted[('Batting', 'XGBoost')].save_model(EXPORT_DIR / 'batting_model.json')
fitted[('Bowling', 'XGBoost')].save_model(EXPORT_DIR / 'bowling_model.json')
win_model.save_model(EXPORT_DIR / 'win_model.json')

# load the win model back and confirm it predicts the same
reloaded = XGBClassifier()
reloaded.load_model(EXPORT_DIR / 'win_model.json')
assert np.allclose(reloaded.predict_proba(Xw_te.iloc[:100]), win_model.predict_proba(Xw_te.iloc[:100]))

for f in sorted(EXPORT_DIR.glob('*.json')):
    print(f'{f.name}: {f.stat().st_size / 1024:.0f} KB')

## 14. Download

In [ ]:
try:
    from google.colab import files
    for name in ['batting_model.json', 'bowling_model.json', 'win_model.json']:
        files.download(f'models_export/{name}')
except ImportError:
    print('Not running in Colab, the files are in the models_export folder.')